# Compare Benchmark Runs
This notebook demonstrates how you can analyze the differences between two benchmark runs of the same benchmark and find the tests that differ the most, which probably means that they require further analysis to figure out why they changed.

Several projects exist in the `examples` folder, but this notebook assumes we are working on the
JVM part of the `kotlin-multiplatform` project. But the same approach can be used for the other projects.

First, you need to run the benchmark twice. This can be done by running these commands from the root of the project:

```shell
> ./gradlew :examples:kotlin-multiplatform:jvmBenchmark
> ./gradlew :examples:kotlin-multiplatform:jvmBenchmark
```

Once it is completed, run this notebook, and it will automatically find the latest result.

In [85]:
%use serialization, dataframe, kandy

In [86]:
// Serialization classes matching the JMH-alike JSON format.
// We define these classes manually so we can keep `params` as a JsonObject, as it means we can handle them
// in a generic manner. If you benchmark have fixed params, using `"<jsonText>".deserializeThis()` is
// faster and easier.

@Serializable
public data class Benchmark(
    public val benchmark: String,
    public val mode: String,
    public val warmupIterations: Int,
    public val warmupTime: String,
    public val measurementIterations: Int,
    public val measurementTime: String,
    public val primaryMetric: PrimaryMetric,
    public val secondaryMetrics: Map<String, PrimaryMetric>,
    public val params: JsonObject? = null
)

@Serializable
public data class PrimaryMetric(
    public val score: Double,
    public val scoreError: Double,
    public val scoreConfidence: List<Double>,
    public val scorePercentiles: Map<String, Double>,
    public val scoreUnit: String,
    public val rawData: List<List<Double>>,
)

In [113]:
import java.nio.file.Files
import java.nio.file.attribute.BasicFileAttributes
import kotlin.io.path.*

val runsDir = notebook.workingDir.resolve("../build/reports/benchmarks/main")
val resultsFile = runsDir.listDirectoryEntries()
    .filter { it.isDirectory() }
    .maxByOrNull { dir -> Files.readAttributes(dir, BasicFileAttributes::class.java).creationTime() }!!
    .resolve("jvm.json")

In [114]:
val json = Json { ignoreUnknownKeys = true }
val allResults = json.decodeFromString<List<Benchmark>>(resultsFile.readText())

// Split method name into (library, operation, scenario)
// e.g. "...JsonSerializationBenchmark.korlibs_encode_simple" -> korlibs / encode / simple
data class TidyRow(
    val library: String,
    val operation: String,
    val scenario: String,
    val label: String,        // "encode / simple"
    val score: Double,
    val errorLow: Double,
    val errorHigh: Double,
)

val scenarioOrder = listOf(
    "simple",
    "nested",
    "list",
    "nullable_full",
    "nullable_nulls",
    "fetchTimeViaNow",
    "thousandTimesMilliseconds",
)

val tidy = allResults.mapNotNull { r ->
    val method = r.benchmark.substringAfterLast('.')  // e.g. "korlibs_encode_simple"
    val parts  = method.split("_", limit = 3)
    if (parts.size < 3) return@mapNotNull null
    val (lib, op, scenario) = parts
    if (lib !in listOf("korlibs", "kotlinx")) return@mapNotNull null
    TidyRow(
        library   = lib,
        operation = op,
        scenario  = scenario,
        label     = "$op / $scenario",
        score     = r.primaryMetric.score,
        errorLow  = r.primaryMetric.scoreConfidence[0],
        errorHigh = r.primaryMetric.scoreConfidence[1],
    )
}.sortedWith(compareBy({ it.operation }, { scenarioOrder.indexOf(it.scenario) }))

println("Parsed ${tidy.size} rows")
tidy.take(6).forEach { println(it) }

Parsed 26 rows
TidyRow(library=korlibs, operation=decode, scenario=simple, label=decode / simple, score=0.5776926145027088, errorLow=0.5080438080044358, errorHigh=0.6473414210009818)
TidyRow(library=kotlinx, operation=decode, scenario=simple, label=decode / simple, score=0.4774890991438118, errorLow=0.44278983420876866, errorHigh=0.5121883640788549)
TidyRow(library=korlibs, operation=decode, scenario=nested, label=decode / nested, score=1.0467751639093135, errorLow=1.007486244616455, errorHigh=1.086064083202172)
TidyRow(library=kotlinx, operation=decode, scenario=nested, label=decode / nested, score=0.9834708534145784, errorLow=0.8996302240121063, errorHigh=1.0673114828170505)
TidyRow(library=korlibs, operation=decode, scenario=list, label=decode / list, score=10.018463429378315, errorLow=7.974736493178163, errorHigh=12.062190365578466)
TidyRow(library=kotlinx, operation=decode, scenario=list, label=decode / list, score=7.270053244361063, errorLow=7.042931568301768, errorHigh=7.4971749

In [115]:
// ── Plot 1: Grouped bar chart, encode + decode side by side ─────────────────
import org.jetbrains.kotlinx.kandy.util.color.Color

val palette = mapOf("korlibs" to Color.RED, "kotlinx" to Color.BLUE)

tidy.groupBy { it.operation }.forEach { (op, rows) ->
    rows.sortedBy { scenarioOrder.indexOf(it.scenario) }
        .toDataFrame()
        .plot {
            barsH {
                x("score") { axis.name = "Average time (µs)" }
                y("label") { axis.name = "" }
                fillColor("library") {
                    scale = categorical(
                        "korlibs" to Color.RED,
                        "kotlinx" to Color.BLUE
                    )
                }
                // Error bars via separate layer isn't in kandy barsH directly,
                // so add them as a point range on the same axes:
            }
            layout {
                title = "${op.replaceFirstChar { it.uppercaseChar() }}: korlibs vs kotlinx"
                size = 800 to ((50 * rows.size) + 120)
            }
        }.also { println(it) } // renders each plot inline
}

Plot(datasets=[NamedData(dataFrame=       score                   label library
 0  0,577693         decode / simple korlibs
 1  0,477489         decode / simple kotlinx
 2  1,046775         decode / nested korlibs
 3  0,983471         decode / nested kotlinx
 4 10,018463           decode / list korlibs
 5  7,270053           decode / list kotlinx
 6  0,961339  decode / nullable_full korlibs
 7  0,681458  decode / nullable_full kotlinx
 8  0,588319 decode / nullable_nulls korlibs
 9  0,526948 decode / nullable_nulls kotlinx
)], layers=[Layer(datasetIndex=0, geom=LetsPlotGeom(name=bar), mappings={Aes(name=x)=PositionalMapping(aes=Aes(name=x), columnID=score, parameters=LetsPlotPositionalMappingParametersContinuous(scale=org.jetbrains.kotlinx.kandy.ir.scale.PositionalDefaultScale@62c8e699, axis=Axis(name=Average time (µs), position=DEFAULT, min=null, max=null, breaks=null, labels=null, format=null, expand=null))), Aes(name=y)=PositionalMapping(aes=Aes(name=y), columnID=label, parameters=

In [116]:
// ── Plot 2: Ratio chart (korlibs ÷ kotlinx) ──────────────────────────────────
data class RatioRow(val label: String, val ratio: Double, val color: String)

val ratios = tidy
    .groupBy { "${it.operation}_${it.scenario}" }
    .mapNotNull { (_, rows) ->
        val k  = rows.firstOrNull { it.library == "korlibs" } ?: return@mapNotNull null
        val kx = rows.firstOrNull { it.library == "kotlinx"  } ?: return@mapNotNull null
        RatioRow(
            label = "${k.operation} / ${k.scenario}",
            ratio = k.score / kx.score,
            color = if (k.score > kx.score) "korlibs slower" else "korlibs faster"
        )
    }
    .sortedByDescending { it.ratio }

ratios.toDataFrame().plot {
    barsH {
        x("ratio") { axis.name = "korlibs ÷ kotlinx  (>1 = korlibs slower)" }
        y("label") { axis.name = "" }
        fillColor("color") {
            scale = categorical(
                "korlibs slower" to Color.RED,
                "korlibs faster" to Color.GREEN
            )
        }
    }
    layout {
        title = "Performance ratio — korlibs vs kotlinx"
        size = 800 to ((50 * ratios.size) + 120)
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="otwfO0"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 800.0, 
 height: 720.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("otwfO0");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Performance ratio — korlibs vs kotlinx"
},
"mapping":{
},
"data":{
"color":["korlibs slower","korlibs slower","korlibs slower","korlibs slower","korlibs slower","korlibs slower","korlibs slower","korlibs slower","korlibs slower","korlibs slower","korlibs slower","korlibs faster"],
"label":["encode / nested","encode / list","encode / nullable_full","encode / nullable_nulls","encode / simple","decode / nullable_full","decode / list","time / thousandTimesMilliseconds","decode / simple","decode / nullable_nulls","decode / nested","time / fetchTimeViaNow"],
"ratio":[2.402615933424861,2.2835601726019763,2.0930336873507875,1.8446403593300649,1.8288430066597727,1.4107086610026336,1.3780453997567386,1.3451665653994198,1.2098550847308818,1.1164640469097362,1.0643682629484592,0.45811006500142326]
},
"ggsize":{
"width":800.0,
"height":720.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"korlibs ÷ kotlinx (>1 = korlibs slower)",
"limits":[null,null]
},{
"aesthetic":"y",
"discrete":true,
"name":""
},{
"aesthetic":"fill",
"values":["#ee6666","#3ba272"],
"limits":["korlibs slower","korlibs faster"]
}],
"layers":[{
"mapping":{
"x":"ratio",
"y":"label",
"fill":"color"
},
"stat":"identity",
"orientation":"y",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
}],
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"ratio"
},{
"type":"str",
"column":"label"
},{
"type":"str",
"column":"color"
}]
},
"spec_id":"17"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 
 
 0.5 
 
 
 
 
 
 
 
 
 1 
 
 
 
 
 
 
 
 
 1.5 
 
 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 
 
 2.5 
 
 
 
 
 
 
 
 
 
 
 encode / nested 
 
 
 
 
 
 
 encode / list 
 
 
 
 
 
 
 encode / nullable_full 
 
 
 
 
 
 
 encode / nullable_nulls 
 
 
 
 
 
 
 encode / simple 
 
 
 
 
 
 
 decode / nullable_full 
 
 
 
 
 
 
 decode / list 
 
 
 
 
 
 
 time / thousandTimesMilliseconds 
 
 
 
 
 
 
 decode / simple 
 
 
 
 
 
 
 decode / nullable_nulls 
 
 
 
 
 
 
 decode / nested 
 
 
 
 
 
 
 time / fetchTimeViaNow 
 
 
 
 
 
 
 
 
 Performance ratio — korl